# MCP Server Tool Probe in `/configure mcp`

**Status:** Approved design; written specification awaiting review
**Design epic:** `bd-mpuvn`
**Source optimizations:** `sol_58adeffe0dcf4243` (trigger/result model, complete), `sol_d0ac9ddebb454c30` (execution owner, complete), `sol_0112466a93474050` (timeout, complete)
**Interim gate proofs:** `sol_f9fdd4b01c044819`, `sol_8d779727dae14928`, `sol_1480291c4e5440d3` (unsat partition), `sol_1cf8cd89cd884022`, `sol_018d2d1264e84d6c`, `sol_20a8f72e702a451c` (sat witnesses)
**Authoring fallback:** Notebook MCP unavailable at authoring time; hand-written notebook, `ns_mermaid` cell authored-but-not-executed. Obligations verified via solve MCP (evidence table); execute the cell with the notebook runner to mint fresh proof hashes at sign-off.


## Problem and grounded evidence

The `/configure mcp` section (shipped in commits e46dfa87b..4ccede39f) manages server entries but never connects to them. Tool visibility today is indirect and after-the-fact: session detail tracks tools only from observed `ToolCall` events (200ms funnel, `crates/spur-acp/src/domain/events.rs:1255`). There is no way to discover what a configured MCP server offers before (or without) enabling it in a brain session.

The building blocks exist: `McpServerEntry { name, enabled, transport: Stdio{command,args,env} | Http{url,headers} }` (`crates/spur-acp/src/config/mod.rs`), the `McpServersPane` (`crates/spur-tui/src/views/mcp_servers_tui.rs`), and the `rmcp` client already in the dependency tree via `spur-mcp`.

Goal: press a key on an entry in `/configure mcp` → connect, MCP `initialize`, `tools/list` → render the advertised tool catalog (name, description, input schema).


## Architecture decision (Z3 Optimize, all termination: complete)

### 1. Trigger/result model — on-demand, ephemeral

Weights: intentional_spawn=5, no_staleness=3, zero_state_surface=3, discoverability=2, simpler_v1=2 (`sol_58adeffe0dcf4243`).

| Candidate | spawn (5) | stale (3) | state (3) | disc (2) | v1 (2) | Total |
|---|---:|---:|---:|---:|---:|---:|
| **A on-demand ephemeral** | 1 | 1 | 1 | 0 | 1 | **13** |
| B on-demand cached | 1 | 0 | 0 | 0 | 0 | 5 |
| C auto on save | 0 | 0 | 0 | 1 | 0 | 2 |

stdio probes execute user-configured commands — spawning must be an explicit keypress. B's cache can layer on later without UX change.

### 2. Execution owner — `spur-mcp` probe module

Weights: reuse_potential=3, orchestrator_surface_min=3, dep_hygiene=2, latency_directness=1 (`sol_d0ac9ddebb454c30`).

| Candidate | reuse (3) | orch (3) | dep (2) | lat (1) | Total |
|---|---:|---:|---:|---:|---:|
| **A spur-mcp module** | 1 | 1 | 1 | 1 | **9** |
| B tui-local | 0 | 1 | 0 | 1 | 4 |
| C orchestrator round-trip | 0 | 0 | 1 | 0 | 2 |

### 3. Timeout — single 10s hard bound (spawn + initialize + tools/list)

Weights: slow_stdio_tolerance=3, bounded_wait=2, practical_midpoint=2 (`sol_0112466a93474050`): t10s=4 vs t30s=3 vs t5s=2. Deliberate sacrifice: 30s tolerates cold `npx` starts better, but probe failure is display-only and user-retryable. Applies to both transports in v1; the bound is a function parameter (`probe_server_with_timeout`) with `probe_server` defaulting to 10s so tests can use ~100ms.

Caveat: results are optimal under the stated weights, not universal proofs.


In [ ]:
flowchart TD
    SPEC["`@spec MCP-PROBE-OUTCOME
@type Outcome = enum[tools_listed, connect_error, timeout]
@input timed_out: Bool
@input succeeded: Bool
@output outcome: Outcome
@requires PRE: true`"]

    OK["`@branch TOOLS_LISTED
@when not timed_out and succeeded
@ensures OK_STATUS: outcome = tools_listed`"]

    ERR["`@branch CONNECT_ERROR
@when not timed_out and not succeeded
@ensures ERR_STATUS: outcome = connect_error`"]

    TMO["`@branch TIMEOUT
@when timed_out
@ensures TMO_STATUS: outcome = timeout`"]

    CHECK["`@verify PROBE_DETERMINISTIC: prove determinism
@verify PROBE_COVERAGE: prove partition_coverage
@verify PROBE_EXCLUSIVE: prove partition_exclusive
@verify OUTCOMES_REACHABLE: witness each outcome`"]

    SPEC --> OK --> CHECK
    SPEC --> ERR --> CHECK
    SPEC --> TMO --> CHECK


## Proof evidence

The gate reduces to: the three guards `timed_out`, `not timed_out and succeeded`, `not timed_out and not succeeded` partition the `(timed_out, succeeded)` valuations (literal-complementary by construction; the queries verify the guard formulas as written, so a copy-paste bug in a guard would produce `sat`).

| Query | solve_id | Status | Meaning |
|---|---|---|---|
| (t and ok) or (t and err) | `sol_f9fdd4b01c044819` | unsat | timeout-branch vs both completion-branches exclusive (disjunction unsat → both disjuncts unsat) |
| ok and err | `sol_8d779727dae14928` | unsat | the two completion branches are exclusive |
| none of the three guards fire | `sol_1480291c4e5440d3` | unsat | coverage: some branch always fires |
| not t and s | `sol_1cf8cd89cd884022` | sat | tools_listed reachable (t=F, s=T) |
| not t and not s | `sol_018d2d1264e84d6c` | sat | connect_error reachable (t=F, s=F) |
| t | `sol_20a8f72e702a451c` | sat | timeout reachable (t=T) |

Method note (carried from the gateway spec): decompose partition proofs into small per-obligation queries; a single hand-built violation tree mis-nests silently. The cell runner re-verifies determinism/coverage/exclusivity/reachability at execution time.


## Components

### 1. `crates/spur-mcp/src/probe.rs`

```rust
pub struct ProbedTool {
    pub name: String,
    pub description: Option<String>,
    pub input_schema_json: String, // raw JSON schema from tools/list
}

pub enum ProbeOutcome {
    ToolsListed(Vec<ProbedTool>),
    ConnectError(String), // spawn failure, HTTP connect failure, initialize rejection
    Timeout,
}

pub struct ProbeReport {
    pub server_name: String,
    pub outcome: ProbeOutcome,
}

pub async fn probe_server(entry: &McpServerEntry) -> ProbeReport; // 10s bound
pub async fn probe_server_with_timeout(entry: &McpServerEntry, timeout: Duration) -> ProbeReport;
```

- stdio: spawn `command` with `args` + `env`, rmcp stdio client transport; http: streamable HTTP with `headers`.
- Hard timeout covers spawn + initialize + tools/list; the spawned child is killed on drop (rmcp transport cancel).
- Pure read: never mutates config, never calls a tool. No `tools/call`.
- Registered as `pub mod probe;` + re-exports in `spur-mcp/src/lib.rs`.

### 2. TUI wiring (`crates/spur-tui/src/views/mcp_servers_tui.rs` + app layer)

- `t` on a list row: entry state `idle → probing` (row renders `probing…`), `tokio::spawn(probe_server(entry))` with a oneshot back to the app event loop; result applied only to the entry that started it.
- Concurrency guard: one probe in flight per entry; pressing `t` again while probing is a no-op (or cancels + restarts — no-op in v1).
- Render: collapsible tool list under the entry row — name + one-line description; expand a tool to view its input schema. Footer keeps the next-session notice. Errors render inline (`connect error: <msg>` / `timeout after 10s`).
- The pane stays synchronous-state; the async runner is injectable (`probe_hook: Option<Arc<dyn Fn(McpServerEntry) -> BoxFuture<ProbeReport>>>`) so pane logic is unit-testable without I/O.

## Task decomposition (for the implementation plan DAG)

- **T1** `spur-mcp`: probe.rs (types, stdio + http paths, timeout, kill-on-drop) + subprocess stdio fixture test + http error-path test. RED-GREEN.
- **T2** `spur-tui`: pane `t` action, probing state, injectable runner, result rendering + tests (depends on T1).

## Testing strategy

- T1: hermetic stdio fixture — a `tests/probe_fixture.rs` binary serving a fixed 2-tool registry over stdio (reuse `RegistryServerBuilder`/`serve_stdio_server`), spawned via `current_exe` + marker env (subprocess pattern already used by `worker_mcp` context-service tests); assert 2 `ProbedTool`s. Error paths: nonexistent command → `ConnectError`; `http://127.0.0.1:1/` → `ConnectError`; 100ms timeout against a fixture that sleeps → `Timeout`.
- T2: pane unit tests with a fake probe hook — probing state renders, ok result renders N tools + expand/collapse, error/timeout render, second `t` while probing is a no-op, result applies to the starting entry only.
- Re-run solver gates against final guard code if branch predicates change.

## Non-goals (v1)

- Caching probe results (Option B) and auto-probe-on-save (Option C).
- Invoking tools (`tools/call`), prompts or resources listing (tools only).
- CLI `spur mcp list-tools` subcommand (falls out nearly free later — same probe module).
- Probe of entries with invalid payloads (validation gate rejects them at persist time anyway).

## Risks

- stdio child leaks on timeout — mitigated by kill-on-drop + fixture test asserting process exit.
- Large tool catalogs (hundreds of tools) — v1 renders a plain scrollable list; paging later if needed.
- HTTP servers requiring OAuth — surfaces as `ConnectError` with the 401 detail; header-token servers work today.
